# LELA60341: Research Methods in Computational and Corpus Linguistics Week 9

## Text as data: a `TextEncoder` class

Last week you built logistic regression from parts, a `Linear` layer, a loss and a training loop, and trained it on examples given as rows of numbers. This week starts by using and evaluating that model. In last week's LELA60331 seminar (*Turning text into features*) you saw how to turn real text, 10,000 Yelp reviews, into rows of numbers: a vocabulary, and one-hot encoding. In the rest of this workbook you write a class that does that job: a `TextEncoder`.

| Part | Topic |
|---|---|
| 1 | Using and evaluating the Week 8 model |
| 2 | The plan |
| 3 | Loading the reviews |
| 4 | The `TextEncoder` class |
| 5 | Encoding the Yelp reviews |
| 6 | A different tokeniser, by inheritance |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · Using and evaluating the Week 8 model

In the Week 8 after-class workbook you trained a logistic regression model, a `Linear` layer with one output, on the simulated data. Before we move on to real text, we use that model, evaluate it, and compare several models side by side.

### Set-up

The next three cells set up everything from Week 8: the simulated data `X` and `y`, the helper functions, the `Linear` class, `bce_loss` and `bce_grad`. The last of them trains the model, as at the end of the Week 8 after-class workbook. Read through them to remind yourself of each piece, then run them: the final loss should be 0.656, as in Week 8.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
X = [[x[0][k], x[1][k]] for k in range(len(y))]

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X                       # kept for the backward pass
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):
            for c in range(n_out):
                for j in range(n_in):
                    self.weight_grad[c][j] += grad[k][c] * self.inputs[k][j]
                self.bias_grad[c] += grad[k][c]
        # the gradients for the inputs, for a layer that comes before this one
        return [mat_vec(transpose(self.weight), g) for g in grad]

    def update(self, lr):
        for c in range(len(self.weight)):
            for j in range(len(self.weight[c])):
                self.weight[c][j] -= lr * self.weight_grad[c][j]
            self.bias[c] -= lr * self.bias_grad[c]

def bce_loss(scores, y):
    # the average cross-entropy loss for two classes, from the scores
    eps = 0.00001
    losses = []
    for k in range(len(y)):
        q = sigmoid(scores[k][0])
        if y[k] == 1:
            losses.append(-math.log2(q + eps))
        else:
            losses.append(-math.log2(1 - q + eps))
    return sum(losses) / len(losses)

def bce_grad(scores, y):
    # the gradient of the loss for each score: the error divided by the number of examples
    n = len(y)
    return [[(sigmoid(scores[k][0]) - y[k]) / n] for k in range(n)]

In [ ]:
random.seed(10)
model = Linear(2, 1)
loss_history = []
for epoch in range(1000):
    scores = model.forward(X)
    loss = bce_loss(scores, y)
    loss_history.append(loss)
    model.backward(bce_grad(scores, y))
    model.update(0.5)
print(model.weight, model.bias)
print("final loss:", round(loss_history[-1], 3))

### Using the model

The model gives **scores**. To use it, we write three small functions. They work for any model that has a `forward` method and gives one score per example, so they are kept outside the class.

### 🏋️ Exercise 1 · Practice — Probabilities and predictions

1. Complete `predict_proba(model, X)`: the probability that each example is positive, the sigmoid of its score. *Hint:* a list comprehension over the rows of `model.forward(X)`; each row holds one score.
2. Complete `predict(model, X)`: the predicted class of each example, 1 if its probability is above 0.5 and 0 otherwise
3. `new_probs`: the probability that each of these two new texts is positive (LELA60331 Week 7, Activity 3). Remember that the features are **log** counts, and that the model takes a **list** of examples.
   * a) 2 positive words and 3 negative words
   * b) 10 positive words and 1 negative word

In [ ]:
def predict_proba(model, X):
    return ???          # 1.

def predict(model, X):
    return ???        # 2.

new_probs = ???    # 3.
print(new_probs)                                 # about 0.466 and 0.961
print(predict(model, X[:10]), y[:10])

In [ ]:
# ✅ Check
check("1. predict_proba", lambda: abs(predict_proba(model, [[0, 0]])[0] - sigmoid(model.bias[0])) < 1e-12)
check("2. predict", lambda: predict(model, X[:5]) == [int(p > 0.5) for p in predict_proba(model, X[:5])])
check("3. new_probs", lambda: abs(new_probs[0] - 0.466) < 0.005 and abs(new_probs[1] - 0.961) < 0.005)

### 🏋️ Exercise 2 · Practice — `evaluate`

Complete `evaluate(model, X, y)`, which returns a **dict** (Week 2) with the accuracy, precision and recall (LELA60331 Week 7, Activities 2 and 4):

1. the predicted classes, with `predict`
2. the conditions for false positives and false negatives (the others are done for you)
3. the three entries of the dict

In [ ]:
def evaluate(model, X, y):
    predicted = ???                         # 1.
    correct = 0
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(y)):
        if predicted[k] == y[k]:
            correct += 1
        if predicted[k] == 1 and y[k] == 1:
            tp += 1
        if ???:               # 2. predicted positive, but really negative
            fp += 1
        if ???:               #    predicted negative, but really positive
            fn += 1
    return {"accuracy": ???, "precision": ???, "recall": ???}     # 3.

results = evaluate(model, X, y)
print(results)                     # accuracy 0.78, precision 0.75, recall 0.783

In [ ]:
# ✅ Check
check("accuracy 0.78, precision 0.75, recall 0.783",
      lambda: results["accuracy"] == 0.78 and results["precision"] == 0.75 and abs(results["recall"] - 0.783) < 0.001)

### Several models side by side

In a notebook written as a series of cells, trying a different learning rate means changing `lr` and re-running the training cell, which replaces the old model. With a class, each model is a separate object, so you can keep several and compare them.

### 🏋️ Exercise 3 · Practice — A `train` function

Complete `train(model, X, y, lr, n_epochs)`, which trains `model` with the loop from Exercise 8 of the Week 8 after-class workbook and returns its loss history. The test underneath checks that it gives the same result as the training cell in the set-up above.

In [ ]:
def train(model, X, y, lr, n_epochs):
    history = []
    for epoch in range(n_epochs):
        scores = ???
        history.append(???)
        ???
        ???
    return history

random.seed(10)
test_model = Linear(2, 1)
test_history = train(test_model, X, y, 0.5, 1000)
print(round(test_history[-1], 3))         # the same as the training loop: 0.656

In [ ]:
# ✅ Check
check("train gives the same as the training loop", lambda: abs(test_history[-1] - loss_history[-1]) < 1e-12 and len(test_history) == 1000)

### 🏋️ Exercise 4 · Practice — Comparing learning rates

1. `models`: a dict from each learning rate in `rates` to a new `Linear(2, 1)` model (made after `random.seed(10)`, so that they all start the same)
2. `histories`: a dict from each learning rate to the loss history from training its model with `train`

The cell then plots the three loss histories on the same axes. 🔮 **Predict** before you run it: what will each curve look like? (Think about what you saw when you changed the learning rate for linear regression in LELA60331 Week 4.)

In [ ]:
rates = [0.01, 0.5, 2]
models = {}
histories = {}
for lr in rates:
    random.seed(10)
    models[lr] = ???                                         # 1.
    histories[lr] = ???                 # 2.

for lr in rates:
    plt.plot(histories[lr], label="lr = " + str(lr))
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
for lr in rates:
    print(lr, round(histories[lr][-1], 3), evaluate(models[lr], X, y))

In [ ]:
# ✅ Check
check("1. three models", lambda: sorted(models) == rates and all(isinstance(m, Linear) for m in models.values()))
check("2. each trained for 1000 epochs", lambda: all(len(h) == 1000 for h in histories.values()))
check("lr = 0.5 gives the same as the training loop", lambda: abs(histories[0.5][-1] - loss_history[-1]) < 1e-9)
check("the first model is unchanged", lambda: abs(bce_loss(model.forward(X), y) - 0.656) < 0.001)

⭐ **Challenge (optional):** add a method `__str__` to `Linear` (Week 7, Part 3), so that `print(model)` shows its weights and biases, rounded to 2 decimal places. *Hint:* copy the class from the set-up cell into a new cell, and add the method.

---
# Part 2 · The plan

In the LELA60331 Week 8 seminar, turning the reviews into features took many cells and many global names: `tokenized_sents`, `counts`, `type_list`, `word_to_index`, `M_train`, `M_test`, and the function `one_hot`. Here they become one object:

| `TextEncoder` | What it is or does |
|---|---|
| `_vocab_size` | instance variable: how many words to keep in the vocabulary |
| `_type_list` | instance variable: the vocabulary words, in column order |
| `_word_to_index` | instance variable: a dict from each vocabulary word to its column |
| `tokenise(text)` | method: split a text into a list of tokens |
| `fit_vocabulary(texts)` | method: build the vocabulary from a list of texts |
| `one_hot_encode(texts)` | method: turn a list of texts into sparse one-hot rows |
| `decode(row)` | method: turn a sparse row back into its words |
| `len(encoder)`, `encoder[j]` | special methods: the size of the vocabulary, and the word for column `j` |

An encoder that has built its vocabulary can then turn any texts into rows, for training or for testing, always with the same columns.

---
# Part 3 · Loading the reviews

The data set is the one from last week's LELA60331 seminar: 10,000 reviews from the website Yelp, each labelled `negative` (1 or 2 stars) or `positive` (3 or 4 stars). The text has already been lower-cased, and its punctuation separated from the words by spaces, so that it is easy to split into words. A line beginning with `!` is a command for the computer, not Python: `wget` downloads the file.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

### 🏋️ Exercise 5 · Practice — `load_reviews(path)`

Write a function that reads the file and returns **two** lists (Week 4, *Returning several values*): the texts, and the labels as numbers, `1` for positive and `0` for negative.

The file is **tab-separated**: each line is a review, a tab (`"\t"`), then the label. The first line is a header, so skip it. Fill in:

1. the two fields of the line. *Hint:* `rstrip()` removes the newline at the end, and `split("\t")` splits at the tab.
2. adding the text to `texts`
3. adding the label to `labels`, as 1 or 0. *Hint:* `int(fields[1] == "positive")`

In [ ]:
def load_reviews(path):
    texts = []
    labels = []
    with open(path) as fp:
        lines = fp.readlines()
    for line in lines[1:]:              # skip the header line
        fields = ???          # 1. [text, label]
        ???                     # 2.
        ??? # 3.
    return texts, labels

texts, labels = load_reviews("yelp_reviews.txt")
print(len(texts), sum(labels))
print(labels[0], texts[0][:60])

In [ ]:
# ✅ Check
check("10000 reviews", lambda: len(texts) == 10000 and len(labels) == 10000)
check("5000 positive", lambda: sum(labels) == 5000 and set(labels) == {0, 1})
check("the first review", lambda: texts[0].startswith("all i can say") and labels[0] == 0)

We want to know how well a classifier does on reviews it has **never seen**, so we split the reviews: 8000 to train on, and 2000 to test on. `random.sample(sequence, k)` picks `k` different elements at random from `sequence`: here, the positions of the training reviews. The test reviews are all the others. With `random.seed(1)` this is exactly the split from the LELA60331 seminar, so your numbers will match. Run the cell.

In [ ]:
import random

random.seed(1)
train_ints = random.sample(range(len(texts)), 8000)
train_set = set(train_ints)
test_ints = [i for i in range(len(texts)) if i not in train_set]

train_texts = [texts[i] for i in train_ints]
train_labels = [labels[i] for i in train_ints]
test_texts = [texts[i] for i in test_ints]
test_labels = [labels[i] for i in test_ints]
print(len(train_texts), len(test_texts), sum(train_labels), sum(test_labels))

---
# Part 4 · The `TextEncoder` class

## Turning text into features

Here is a reminder of how we turn texts into features (LELA60331 Week 8). Logistic regression needs every text to be a list of numbers, its **features**. In Week 8 of this workbook, the simulated texts had just two (the log counts of positive and of negative words). For real text we use one feature for every word in a **vocabulary**: the feature is 1 if the word occurs in the text and 0 if it doesn't. This is called **one-hot encoding**.

* The **vocabulary** is the list of words we use as features. We take the most frequent words in the corpus (5000 of them for the Yelp reviews). Each word has a fixed position in the list, its **column**.
* Each text becomes a **row** with one number per column. Here is a vocabulary of 4 words, and the row for the text *"good food good"*:

| column | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| word | good | bad | food | service |
| *"good food good"* | 1 | 0 | 1 | 0 |

"good" occurs twice, but its column still holds a 1: we only record whether a word is **present**.

With 5000 words, almost every number in a row is 0: a review contains only a small fraction of the vocabulary. Storing all those 0s would make 8000 reviews take 40 million numbers. So we use a **sparse** representation instead: for each text we store only the **list of columns that hold a 1**. The row above becomes `[0, 2]`.

## The class

Here is what each part of the class does. Read through all of it, then complete the class in Exercise 6.

**`__init__(self, vocab_size=5000)`.** `vocab_size=5000` is a **default value** (Week 8): if you write `TextEncoder()`, `vocab_size` is 5000, and if you write `TextEncoder(100)` or `TextEncoder(vocab_size=100)`, it is 100. The constructor stores it, and starts with an empty vocabulary: an empty list `_type_list` and an empty dict `_word_to_index`.

**`tokenise(self, text)`.** Splits a text into tokens at the spaces. This is done for you. It is a method of its own, rather than a `split()` inside the other methods, so that a subclass can change how *every* method tokenises by overriding just this one (Part 6).

**`fit_vocabulary(self, texts)`.** Builds the vocabulary from the most frequent words in `texts`:

* count every token with a counting dict (Week 3). Note that it calls **another method of the same object**, `self.tokenise(text)`
* make a list of `(count, word)` pairs and sort it from largest to smallest with `sorted(..., reverse=True)`. Python sorts pairs by their first element, so the most frequent words come first
* keep the words from the first `self._vocab_size` pairs in `self._type_list`
* fill in `self._word_to_index`, so that `self._word_to_index[word]` is the column of `word`

**`__len__(self)` and `__getitem__(self, j)`** (Week 7, Part 3). `len(encoder)` gives the number of words in the vocabulary, and `encoder[j]` the word for column `j`.

**`index(self, word)`.** The column of `word`, or `None` if it isn't in the vocabulary. Done for you.

**`one_hot_encode(self, texts)`.** For each text, its sparse row: the sorted list of the columns that hold a 1. Words that are not in the vocabulary are skipped. The columns are collected first in a **set** (Week 4), so that a word that occurs several times is stored only once.

**`decode(self, row)`.** The opposite of encoding: turns a sparse row (a list of columns) back into a list of words. *Hint:* the object's own `[]` works inside its methods too: `self[j]` is the word for column `j`.

### 🏋️ Exercise 6 · Practice — Complete `TextEncoder`

Fill in the blanks 1–6. The cell won't run until every `???` is replaced, so work through them in order. The tests underneath use a tiny corpus, so you can check the answers by hand.

⭐ **Challenge (optional):** add a `__str__` method, so that `print(encoder)` shows something like `TextEncoder with 4 words: good, food, was, the`.

In [ ]:
class TextEncoder:
    """Turn texts into sparse one-hot rows over a vocabulary of the most frequent words."""

    def __init__(self, vocab_size=5000):
        self._vocab_size = vocab_size
        self._type_list = ???                                  # 1. an empty vocabulary
        self._word_to_index = ???

    def tokenise(self, text):
        """Split a text into a list of tokens."""
        return text.split()

    def fit_vocabulary(self, texts):
        """Build the vocabulary from the most frequent tokens in texts."""
        counts = {}
        for text in texts:
            for word in self.tokenise(text):
                counts[word] = ???         # 2. count the word
        pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
        self._type_list = ???   # 3.
        self._word_to_index = {}
        for j in range(len(self._type_list)):
            ???        # 4. word -> column

    def __len__(self):
        return len(self._type_list)

    def __getitem__(self, j):
        return self._type_list[j]

    def index(self, word):
        """The column of word, or None if it is not in the vocabulary."""
        return self._word_to_index.get(word)

    def one_hot_encode(self, texts):
        """A list with one sorted list of columns per text."""
        M = []
        for text in texts:
            row = set()
            for word in self.tokenise(text):
                if ???:                # 5. only vocabulary words
                    row.add(self._word_to_index[word])
            M.append(sorted(row))
        return M

    def decode(self, row):
        """The words for the columns in a sparse row."""
        return ???                          # 6.

    # ⭐ Challenge: __str__
    # def __str__(self):
    #     return ???


tiny = ["the food was good", "the service was bad", "good food good service", "bad food"]
enc = TextEncoder(vocab_size=4)
enc.fit_vocabulary(tiny)
print(len(enc), [enc[j] for j in range(len(enc))])
print(enc.index("food"), enc.index("service"))

🔮 **Predict:** the vocabulary is `good, food, was, the` (columns 0 to 3). What will `enc.one_hot_encode(["good good food", "terrible service"])` give? Write down your answer, then run the cell.

In [ ]:
rows = enc.one_hot_encode(["good good food", "terrible service"])
print(rows)
print(enc.decode(rows[0]))

In [ ]:
# ✅ Check
def _enc_separate():
    a, b = TextEncoder(2), TextEncoder(2)
    a.fit_vocabulary(["x y"])
    return len(a) == 2 and len(b) == 0
check("1. a new encoder has an empty vocabulary", lambda: len(TextEncoder()) == 0 and TextEncoder()._vocab_size == 5000)
check("2–3. the 4 most frequent words", lambda: [enc[j] for j in range(len(enc))] == ["good", "food", "was", "the"])
check("4. word_to_index", lambda: enc.index("food") == 1 and enc.index("the") == 3 and enc.index("service") is None)
check("5. one_hot_encode", lambda: rows == [[0, 1], []])
check("5. each column once, in order", lambda: enc.one_hot_encode(["the was food the good"]) == [[0, 1, 2, 3]])
check("6. decode", lambda: enc.decode([0, 1]) == ["good", "food"] and enc.decode([]) == [])
check("each encoder has its own vocabulary", _enc_separate)
check("⭐ challenge: __str__", lambda: str(enc) == "TextEncoder with 4 words: good, food, was, the")

"service" occurs twice in the tiny corpus, as often as "was" and "the", but it didn't make the top 4. When counts are tied, sorting the `(count, word)` pairs from largest to smallest puts the words in reverse alphabetical order, so "service" lost out.

---
# Part 5 · Encoding the Yelp reviews

Now we use the class on the real data, doing in a few lines what took several cells in the LELA60331 Week 8 seminar.

### 🏋️ Exercise 7 · Practice — Using the encoder

1. Make `encoder`, a `TextEncoder` with a vocabulary of 5000 words, and build its vocabulary from **all** the reviews in `texts`. (This is what the LELA60331 seminar does. Strictly, it would be better to use only the training reviews: see the ⭐ question below.)
2. `M_train` and `M_test`: the one-hot rows of the training texts and of the test texts
3. `average_ones`: the average number of 1s in a row of `M_train`, that is, the average length of a sparse row (LELA60331 Week 8, Activity 7)
4. `first_words`: the first 10 words of the first training review, according to the encoder. *Hint:* decode row 0 of `M_train`, then slice.

Encoding takes a few seconds.

In [ ]:
encoder = ???                       # 1.
???

M_train = ???    # 2.
M_test = ???

average_ones = ???     # 3.
first_words = ???                       # 4.

print(len(encoder), encoder[0], encoder[1])
print(len(M_train), len(M_test), round(average_ones, 1))
print(first_words)

In [ ]:
# ✅ Check
check("1. 5000 words, most frequent first", lambda: len(encoder) == 5000 and encoder[0] == "." and encoder[1] == "the")
check("2. M_train and M_test", lambda: len(M_train) == 8000 and len(M_test) == 2000)
check("2. the first training row", lambda: M_train[0][:10] == [0, 1, 3, 4, 5, 6, 7, 8, 10, 17])
check("3. average_ones", lambda: abs(average_ones - 78.5) < 0.01)
check("4. first_words", lambda: first_words == ['.', 'the', 'and', 'i', 'to', 'a', 'was', 'it', 'is', 'they'])

`first_words` doesn't look much like the start of a review. Why not? What has the one-hot encoding thrown away?

⭐ **Challenge (optional):** we built the vocabulary from all 10,000 reviews, including the test reviews. Why is this, strictly speaking, cheating? Make a second encoder whose vocabulary is built from `train_texts` only. How many of its 5000 words are different? *Hint:* make a set of each encoder's words.

In [ ]:
# ⭐ Challenge

---
# Part 6 · A different tokeniser, by inheritance

How we tokenise affects everything else: the vocabulary, and so the features. In a notebook written as a series of cells, trying a different tokenisation means going back, editing the tokenisation cell, and re-running everything below it, losing the old version. With a class, you can instead write a **subclass** that overrides `tokenise`, and keep both versions.

Very frequent function words like *the* and *and* take up many places in our vocabulary. In Week 7 you wrote a `StopwordVocabulary` that ignores **stop words**. Now we do the same for the encoder.

### 🏋️ Exercise 8 · Practice — `StopwordEncoder`

Complete `StopwordEncoder`, a subclass of `TextEncoder`:

1. the constructor calls the parent's constructor with `vocab_size` (so that the instance variables are created), then stores the stop words in a **set** in `self._stopwords`
2. `tokenise(text)` calls the **parent's** `tokenise` with `super()`, then keeps only the tokens that are not stop words. *Hint:* a list comprehension with an `if`.

You don't need to write `fit_vocabulary` or `one_hot_encode` again.

In [ ]:
class StopwordEncoder(TextEncoder):
    """A TextEncoder that ignores stop words."""

    def __init__(self, stopwords, vocab_size=5000):
        ???                   # 1.
        self._stopwords = ???

    def tokenise(self, text):
        return ???   # 2.

stop = StopwordEncoder(["the", "a", "and", "was"], vocab_size=4)
print(stop.tokenise("the food was good and the service was bad"))
stop.fit_vocabulary(tiny)
print([stop[j] for j in range(len(stop))])

In [ ]:
# ✅ Check
check("1. still a TextEncoder", lambda: isinstance(stop, TextEncoder) and stop._vocab_size == 4)
check("2. tokenise drops stop words", lambda: stop.tokenise("the food was good and the service was bad") == ["food", "good", "service", "bad"])
check("fit_vocabulary uses the new tokenise", lambda: stop.index("the") is None and stop.index("service") is not None)
check("one_hot_encode uses the new tokenise", lambda: stop.one_hot_encode(["the bad food"]) == [sorted([stop.index("bad"), stop.index("food")])])

🔮 **Predict:** `fit_vocabulary` and `one_hot_encode` belong to `TextEncoder`, and you didn't change them. Yet they now skip the stop words. Why? (Think about Week 7, Exercise 5: when `fit_vocabulary` calls `self.tokenise(text)`, which class's version runs, and why?)

Finally, here is a stop-word encoder for the Yelp reviews. Run the cell. Which words have taken the places of the stop words?

In [ ]:
yelp_stop = StopwordEncoder([".", ",", "the", "and", "i", "to", "a", "was", "it", "of"])
yelp_stop.fit_vocabulary(texts)
print([yelp_stop[j] for j in range(15)])
print(yelp_stop.index("the"), encoder.index("the"))     # each encoder has its own vocabulary

---
## ✏️ Summary

You can now:

* use a trained model to give probabilities and predictions for new examples
* evaluate a model with a function that works for any model with a `forward` method
* wrap the training loop in a function, and use it to keep several models side by side and compare them
* write a `TextEncoder` class that builds a vocabulary and one-hot encodes texts, with `len()` and `[]` support
* call one method from another with `self.`, and override a single method in a subclass to change the behaviour of the others